In [1]:
# %% [markdown]
# # 6 注意力机制
# ## 6.1 理论计算题：缩放点积注意力手工计算
# Q ∈ R^{2×4}, K ∈ R^{3×4}, V ∈ R^{3×5}, d_k=4
# 步骤：
# 1. Score = Q @ K^T / sqrt(d_k) → shape (2,3)
# 2. Attn_weight = softmax(Score) 逐行softmax
# 3. Output = Attn_weight @ V → shape (2,5)
#
# ## 6.2 编程题：多头注意力 num_heads=2, d_model=4
# %%
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        # 投影矩阵 Q,K,V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        # 输出融合线性层
        self.w_o = nn.Linear(d_model, d_model)
    
    def scaled_dot_product(self, q, k, v):
        dk = q.size(-1)
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(dk, dtype=torch.float32))
        attn_weight = F.softmax(attn_score, dim=-1)
        out = torch.matmul(attn_weight, v)
        return out
    
    def split_heads(self, x):
        # x: (seq_len, batch, d_model)
        seq_len, batch, _ = x.shape
        return x.view(seq_len, batch, self.num_heads, self.d_k).transpose(1,2) # (seq_len, head, batch, dk)
    
    def concat_heads(self, x):
        seq_len, head, batch, dk = x.shape
        return x.transpose(1,2).contiguous().view(seq_len, batch, self.d_model)
    
    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        # 线性投影
        q = self.w_q(X)
        k = self.w_k(X)
        v = self.w_v(X)
        # 分头
        q_head = self.split_heads(q)
        k_head = self.split_heads(k)
        v_head = self.split_heads(v)
        # 单头注意力
        attn_out = self.scaled_dot_product(q_head, k_head, v_head)
        # 拼接多头
        concat = self.concat_heads(attn_out)
        # 输出线性层
        final_out = self.w_o(concat)
        return final_out

# 测试
if __name__ == "__main__":
    seq_len, batch = 6, 2
    d_model = 4
    mha = MultiHeadAttention(d_model=d_model, num_heads=2)
    x = torch.randn(seq_len, batch, d_model)
    res = mha(x)
    print("多头注意力输出 shape:", res.shape) # (6,2,4) 和输入一致

多头注意力输出 shape: torch.Size([6, 2, 4])
